---
title: "Chapter 12: Results and Deduplication"
short_title: "12. Results"
---

This chapter presents the numbers required for Milestone 2: the **parsing success rate**, the **number of elements
and references before and after deduplication**, and the three output Parquet tables.

## 1. Element IDs and content deduplication (`dedup_elements.py`)

Each node receives an `element_id` of the form `<yymm-id>#<kind><number>`; for example `2601-13156#s42` is the 42nd
sentence ID assigned in paper 2601.13156. The ID contains the paper ID, so IDs never collide across papers. Content
is compared **exactly, after whitespace normalization** (no case folding, no approximate matching):

- Sentences, formulas and figures that are **identical** at corresponding occurrences across versions **share one ID**.
- The same content appearing a second time **within one version** gets a new ID, so that every node has exactly one
  parent and the `hierarchy` table remains a tree.
- Repeated headings are told apart by order of appearance; structural nodes (lists, items) by parent and order; the
  root of each version has its own content `Version n root`.

**Before deduplication** is the number of node occurrences (each edge of `hierarchy` plus one root per version);
**after** is the number of rows of the `elements` table. The reduction is a reduction in **rows**, not in file size.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils_m2 import (COLORS, KIND_LABELS, PROBLEM_LABELS, group, human_bytes, human_seconds, load,
                             members, meta, metric, metric_table, num, ordered, pct, percent, ready, secs,
                             show, table)
READY = ready()

## 2. Parsing success rate

In [ ]:
if READY:
    names = {"Papers found in Milestone 1": "discovered_papers_selected", "Papers with a parse result": "papers_with_output",
             "Papers not parsed": "papers_not_parsed", "Papers with a missing or invalid result": "papers_missing_or_invalid_output"}
    t = metric_table(names)
    row = {"Metric": "Parsing success rate"}
    for mm in members()["member"]:
        row[mm] = percent(metric("parsing_success_percent", mm))
    row["Group"] = pct(metric("papers_with_output") / metric("discovered_papers_selected"))
    t = pd.concat([t, pd.DataFrame([row])], ignore_index=True)
    table(t, "Table 12.1. Parsing success rate per range")

The denominator is the set of papers found in Milestone 1 (IDs that do not exist on arXiv are not counted). A paper
is **parsed successfully** when all its result files in `parsed/` are present and valid, including papers whose
versions have no TeX (their tree has only the root).

## 3. Elements before and after deduplication

In [ ]:
if READY:
    m, g = members(), group()
    allrows = pd.concat([m, g.to_frame().T], ignore_index=True)
    for c in ["element_before", "element_after", "element_reduction_percent", "hierarchy_edges", "bibtex_before",
              "bibtex_after", "bibtex_reduction_percent", "papers", "versions", "versions_with_tex"]:
        allrows[c] = pd.to_numeric(allrows[c], errors="coerce")
    view = pd.DataFrame({
        "Range": allrows["member"].replace({"group": "Group"}),
        "Papers": allrows["papers"].map(num), "Versions": allrows["versions"].map(num),
        "Elements before": allrows["element_before"].map(num), "Elements after": allrows["element_after"].map(num),
        "Reduction (elements)": allrows["element_reduction_percent"].map(percent),
        "Hierarchy edges": allrows["hierarchy_edges"].map(num),
        "BibTeX before": allrows["bibtex_before"].map(num), "BibTeX after": allrows["bibtex_after"].map(num),
        "Reduction (BibTeX)": allrows["bibtex_reduction_percent"].map(percent)})
    table(view, "Table 12.2. Elements and BibTeX entries before and after deduplication")
    long = allrows[allrows["member"] != "group"].melt(id_vars=["member"], value_vars=["element_before", "element_after"],
                                                       var_name="k", value_name="n")
    long["Count"] = long["k"].map({"element_before": "Before (occurrences)", "element_after": "After (distinct elements)"})
    fig = px.bar(long, x="member", y="n", color="Count", barmode="group", color_discrete_sequence=COLORS,
                 title="Elements before and after deduplication per range", labels={"member": "Range", "n": "Elements"})
    show(fig)

The group's reductions are recomputed from the summed counts of the five ranges (`group_benchmark.py --m2`), not
averaged over the percentages.

## 4. Deduplication by number of versions

The benefit of shared IDs comes from papers with several versions: content unchanged between `v1` and `v2` is stored
only once. The chart below groups papers by their number of versions:

In [ ]:
if READY:
    p = load("papers")
    p = p[p["status"] == "done"].copy()
    for c in ["edges", "versions", "elements"]:
        p[c] = pd.to_numeric(p[c], errors="coerce")
    p["before"] = p["edges"] + p["versions"]
    p["Versions"] = p["versions"].map(lambda n: str(int(n)) if n < 4 else "≥ 4")
    gp = p.groupby("Versions").agg(papers=("arxiv_id", "size"), before=("before", "sum"), after=("elements", "sum")).reset_index()
    gp["Reduction (%)"] = 100 * (1 - gp["after"] / gp["before"])
    fig = px.bar(gp, x="Versions", y="Reduction (%)", text=gp["Reduction (%)"].map(lambda x: f"{x:.1f}%"),
                 color_discrete_sequence=COLORS, title="Element reduction by number of versions of the paper")
    show(fig)
    table(pd.DataFrame({"Versions": gp["Versions"], "Papers": gp["papers"].map(num), "Elements before": gp["before"].map(num),
                        "Elements after": gp["after"].map(num), "Reduction": gp["Reduction (%)"].map(percent)}),
          "Table 12.3. Deduplication by number of versions")

Papers with a single version barely shrink: content repeated within one version still gets its own ID to keep the
tree structure correct. The group's reduction comes from papers with two or more versions, and grows with the
number of versions.

## 5. The three output Parquet tables

In [ ]:
if READY:
    t = load("tables")
    t = ordered(t)
    t["member"] = t["member"].replace({"merged": "Merged (group)"})
    view = pd.DataFrame({"Range": t["member"], "Papers": t["papers"].map(num)})
    for k in ["elements", "hierarchy", "bibtex"]:
        view[f"{k}: rows"] = pd.to_numeric(t[f"{k}_rows"], errors="coerce").map(num)
        view[f"{k}: size"] = pd.to_numeric(t[f"{k}_bytes"], errors="coerce").map(human_bytes)
    table(view, "Table 12.4. Rows and size (Parquet, zstd) of the three tables")
    cf, act = metric("elements_counterfactual_without_dedup_parquet_bytes"), metric("elements_actual_with_dedup_parquet_bytes")
    if pd.notna(cf) and pd.notna(act) and cf > 0:
        print(f"Counterfactual estimate: without deduplication the elements table would take {human_bytes(cf)}, "
              f"actually {human_bytes(act)} ({pct(1 - act / cf)} smaller).")

| Table | One row is | Columns |
| :--- | :--- | :--- |
| `elements` | one distinct element of a paper, across all versions | `element_id`, `arxiv_id`, `kind`, `content` |
| `hierarchy` | one child–parent edge of one version | `arxiv_id`, `version`, `element_id`, `parent_id` |
| `bibtex` | one reference after deduplication | `arxiv_id`, `citation_key`, `entry_type`, `title`, `authors`, `year`, `doi`, `journal`, `url` |

These three tables are the input of Milestone 3 and are uploaded to Hugging Face with the other tables of the
dataset.